In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('imports OK')

---
## Problem 1: Feed Ranking at Scale

### Business Context
Rank thousands of candidate posts for each user's feed. Goal: maximize engagement (CTR, dwell time, shares) while maintaining content diversity and avoiding filter bubbles.

**Scale**: 500M DAU, 100k posts/user pool, <50ms SLA.

### Architecture — Multi-Stage Pipeline
```
User request
    |
    v
[Stage 1] Candidate Retrieval      ~10k candidates, <10ms
    Two-tower ANN, collaborative filtering, content-based
    |
    v
[Stage 2] Light Ranking             ~1k candidates, <15ms
    GBDT on sparse features, fast inference
    |
    v
[Stage 3] Heavy Ranking             ~200 candidates, <20ms
    Deep neural network, cross-features, user history
    |
    v
[Stage 4] Re-ranking                ~20 items, <5ms
    Diversity (MMR), freshness boost, business rules
```

### Key Concepts

**1. Learning to Rank (LTR) approaches:**
- **Pointwise**: score each item independently (CTR model). No relative order in loss.
- **Pairwise**: penalize inversions between item pairs (RankNet, LambdaRank). Better than pointwise.
- **Listwise**: optimize over the full ranked list (LambdaMART). Closest to NDCG. Industry standard.

**2. NDCG (Normalized Discounted Cumulative Gain):**
```
DCG@k  = Σ relevance_i / log2(position_i + 1)  for i in 1..k
NDCG@k = DCG@k / IDCG@k
```
Position discount: lower-ranked items contribute less. NDCG=1 means perfect ranking.

**3. Position Bias:** Users click top items more regardless of quality. Training on raw clicks conflates relevance with position. Fix: IPS debiasing or position-as-feature (zero at inference).

**4. Feedback Loops:** Deployed model controls what users see → what gets clicked → training data. Over time, popular items get more exposure. Fix: exploration (epsilon-greedy), holdout sets, counterfactual logging.

**5. Diversity (MMR — Maximal Marginal Relevance):**
```
score(i | S) = λ * relevance(i) - (1-λ) * max_{j in S} similarity(i, j)
```
Greedily select items that are relevant but dissimilar to already-selected set S.

In [ ]:
# --- NDCG computation ---
def dcg_at_k(relevances, k):
    """Discounted Cumulative Gain at k."""
    relevances = np.array(relevances[:k], dtype=float)
    if len(relevances) == 0:
        return 0.0
    positions = np.arange(1, len(relevances) + 1)
    return np.sum(relevances / np.log2(positions + 1))

def ndcg_at_k(predicted_order, true_relevances, k):
    """NDCG@k: predicted_order is list of item indices sorted by model score."""
    reordered = [true_relevances[i] for i in predicted_order]
    dcg = dcg_at_k(reordered, k)
    ideal = dcg_at_k(sorted(true_relevances, reverse=True), k)
    return dcg / ideal if ideal > 0 else 0.0

# Example
true_relevances = [3, 0, 2, 1, 3, 0, 1, 2]   # ground truth relevance per item
model_ranking   = [0, 4, 2, 7, 3, 1, 6, 5]    # items sorted by model score (descending)
print(f'NDCG@3: {ndcg_at_k(model_ranking, true_relevances, k=3):.4f}')
print(f'NDCG@5: {ndcg_at_k(model_ranking, true_relevances, k=5):.4f}')

# --- MMR re-ranking for diversity ---
def mmr_rerank(candidates, embeddings, relevance_scores, lam=0.7, k=5):
    """
    Maximal Marginal Relevance.
    candidates: list of item indices
    embeddings: (n_items, d) array
    relevance_scores: (n_items,) array
    lam: trade-off relevance vs diversity (1.0 = pure relevance)
    """
    selected = []
    remaining = list(candidates)

    for _ in range(min(k, len(remaining))):
        if not selected:
            # First item: pick highest relevance
            best = max(remaining, key=lambda i: relevance_scores[i])
        else:
            sel_embs = embeddings[selected]
            def mmr_score(i):
                rel = relevance_scores[i]
                sim = np.max(embeddings[i] @ sel_embs.T)  # cosine if normalized
                return lam * rel - (1 - lam) * sim
            best = max(remaining, key=mmr_score)
        selected.append(best)
        remaining.remove(best)

    return selected

# Synthetic example
n_items, d = 20, 8
embeddings = np.random.randn(n_items, d)
embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)  # normalize
relevance_scores = np.random.rand(n_items)

greedy_top5  = list(np.argsort(-relevance_scores)[:5])
diverse_top5 = mmr_rerank(list(range(n_items)), embeddings, relevance_scores, lam=0.6, k=5)

print(f'\nGreedy top-5 (relevance only): {greedy_top5}')
print(f'MMR top-5   (diverse):         {diverse_top5}')
print(f'Avg relevance - greedy: {np.mean(relevance_scores[greedy_top5]):.3f}')
print(f'Avg relevance - MMR:    {np.mean(relevance_scores[diverse_top5]):.3f}')

### L6 Interview Q&A

**Q: Why not just train a pointwise CTR model and sort by it?**  
A: Pointwise models score each item independently — the loss function has no notion of relative order. LambdaMART directly optimizes NDCG by computing gradients from pairwise comparisons weighted by the NDCG gain of swapping each pair. In practice, listwise approaches outperform pointwise by 2-5% NDCG in large-scale feed systems.

**Q: What happens to a ranking model trained on logged click data?**  
A: Position bias inflates CTR for top-ranked items. The model learns to rank items that appeared in position 1 higher — not because they're better, but because they were shown there. Fix: IPS weighting (divide loss by P(shown at position k)), or include position as a feature at training time but zero it out at inference.

**Q: Describe the feedback loop problem and how to address it.**  
A: The deployed model determines what users see → clicks on those items → training data. Items never shown never get clicks, so the model never learns about them. Over time, the item distribution in training data collapses to a narrow set. Fixes: epsilon-greedy exploration (show random items 5% of the time), holdout sets not influenced by the model, counterfactual logging with propensity scores.

### Common Pitfalls
- Using offline NDCG as a proxy for business metrics without online validation
- Forgetting position bias when training on click logs
- Re-ranking at Stage 4 without considering Stage 3 score consistency
- Not logging propensity scores at serving time (needed for IPS debiasing)